### ✅ INGEST THE DATA FROM RDBMS (MYSQL & MSSQL) USING MEDLLIEAN ARCHITECTURE

In [1]:
from IPython.display import display,HTML
display(HTML("<style>pre { white-space: pre !important; } </style>"))

In [2]:
!python3 -V

Python 3.11.6


In [3]:
!which spark-submit

/usr/local/spark/bin/spark-submit


In [4]:
!java --version

openjdk 17.0.15 2025-04-15
OpenJDK Runtime Environment (build 17.0.15+6-Ubuntu-0ubuntu122.04)
OpenJDK 64-Bit Server VM (build 17.0.15+6-Ubuntu-0ubuntu122.04, mixed mode, sharing)


In [5]:
!pip list

Package                       Version
----------------------------- ------------
alembic                       1.12.0
altair                        5.1.2
anyio                         4.0.0
argon2-cffi                   23.1.0
argon2-cffi-bindings          21.2.0
arrow                         1.3.0
asttokens                     2.4.0
async-generator               1.10
async-lru                     2.0.4
attrs                         23.1.0
Babel                         2.13.0
backcall                      0.2.0
backports.functools-lru-cache 1.6.5
beautifulsoup4                4.12.2
bleach                        6.1.0
blinker                       1.6.3
bokeh                         3.3.0
boltons                       23.0.0
Bottleneck                    1.3.7
Brotli                        1.1.0
cached-property               1.5.2
certifi                       2023.7.22
certipy                       0.1.3
cffi                          1.16.0
charset-normalizer            3.3.0
click   

### Now use spark session entry point for all the apis to treat with the different approaches , using `dataframe`

In [6]:
from pyspark.sql import SparkSession

In [7]:
from pyspark.sql.functions import *

In [8]:
from pyspark.sql.types import *

In [9]:
import pyspark.sql.functions as fn 

In [10]:
spark = SparkSession.builder\
.appName('Ingest_data')\
.master('local[*]')\
.config('spark.driver.name' , 'ingest_data') \
.config('spark.driver.host','172.18.0.12')\
.config('spark.sql.catalogImplementation','hive')\
.config('spark.jars','/usr/local/spark-3.5.0-bin-hadoop3/jars/mysql-connector-java-9.3.0.jar,'+
                     '/usr/local/spark-3.5.0-bin-hadoop3/jars/mssql-jdbc-12.6.1.jre8.jar,'+
                     '/usr/local/spark-3.5.0-bin-hadoop3/jars/postgresql-42.7.5.jar')\
.getOrCreate()

In [11]:
spark

In [12]:
sc = spark.sparkContext

In [13]:
sc

<SparkContext master=local[*] appName=Ingest_data>

In [14]:
sc.setLogLevel('ERROR')

#### ✅ Ingest customer information from mysql db 

In [15]:
ingest_mysql_db_cust_info = spark.read.format('jdbc')\
.option('url' , 'jdbc:mysql://mysql_db:3306/DWH_Test') \
.option('dbtable' , 'cust_info') \
.option("driver", "com.mysql.cj.jdbc.Driver") \
.option('user' , 'root') \
.option('password' , 'P@ssw0rd')\
.load()

In [16]:
ingest_mysql_db_cust_info.printSchema()

root
 |-- cst_id: string (nullable = true)
 |-- vst_key: string (nullable = true)
 |-- cst_firstname: string (nullable = true)
 |-- cst_lastname: string (nullable = true)
 |-- cst_marital_status: string (nullable = true)
 |-- cst_gndr: string (nullable = true)
 |-- cst_create_date: string (nullable = true)



In [17]:
ingest_mysql_db_cust_info.columns

['cst_id',
 'vst_key',
 'cst_firstname',
 'cst_lastname',
 'cst_marital_status',
 'cst_gndr',
 'cst_create_date']

In [18]:
ingest_mysql_db_cust_info.dtypes

[('cst_id', 'string'),
 ('vst_key', 'string'),
 ('cst_firstname', 'string'),
 ('cst_lastname', 'string'),
 ('cst_marital_status', 'string'),
 ('cst_gndr', 'string'),
 ('cst_create_date', 'string')]

In [19]:
ingest_mysql_db_cust_info.show(5 , truncate=False)

+------+----------+-------------+------------+------------------+--------+---------------+
|cst_id|vst_key   |cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|
+------+----------+-------------+------------+------------------+--------+---------------+
|11000 |AW00011000| Jon         |Yang        |M                 |M       |2025-10-06\r   |
|11001 |AW00011001|Eugene       |Huang       |S                 |M       |2025-10-06\r   |
|11002 |AW00011002|Ruben        | Torres     |M                 |M       |2025-10-06\r   |
|11003 |AW00011003|Christy      |  Zhu       |S                 |F       |2025-10-06\r   |
|11004 |AW00011004| Elizabeth   |Johnson     |S                 |F       |2025-10-06\r   |
+------+----------+-------------+------------+------------------+--------+---------------+
only showing top 5 rows



In [20]:
ingest_mysql_db_edit_date = ingest_mysql_db_cust_info.withColumn('cst_create_date' , to_date(col('cst_create_date')).alias('cst_create_date'))

In [21]:
ingest_mysql_db_edit_date.show()

+------+----------+-------------+------------+------------------+--------+---------------+
|cst_id|   vst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|
+------+----------+-------------+------------+------------------+--------+---------------+
| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|
| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|
| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|
| 11003|AW00011003|      Christy|         Zhu|                 S|       F|     2025-10-06|
| 11004|AW00011004|    Elizabeth|     Johnson|                 S|       F|     2025-10-06|
| 11005|AW00011005|        Julio|        Ruiz|                 S|       M|     2025-10-06|
| 11006|AW00011006|        Janet|     Alvarez|                 S|       F|     2025-10-06|
| 11007|AW00011007|        Marco|       Mehta|                 M|       M|     2025-10-06|

In [22]:
ingest_mysql_db_edit_date.dtypes


[('cst_id', 'string'),
 ('vst_key', 'string'),
 ('cst_firstname', 'string'),
 ('cst_lastname', 'string'),
 ('cst_marital_status', 'string'),
 ('cst_gndr', 'string'),
 ('cst_create_date', 'date')]

#### ✅ Ingest customer information from mssql db 

In [23]:
ingest_mssql_server_db_prd_info = spark.read.format('jdbc') \
.option('url','jdbc:sqlserver://mssql_server:1433;databaseName=Dwhtest;encrypt=true;trustServerCertificate=true') \
.option('dbtable' , 'dbo.product_info') \
.option('user','sa')\
.option('password' , 'hazemomar95!')\
.option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
.load()

In [24]:
ingest_mssql_server_db_prd_info.printSchema()

root
 |-- prd_id: integer (nullable = true)
 |-- prd_key: string (nullable = true)
 |-- prd_nm: string (nullable = true)
 |-- prd_cost: integer (nullable = true)
 |-- prd_line: string (nullable = true)
 |-- prd_start_dt: date (nullable = true)
 |-- prd_end_dt: date (nullable = true)



In [25]:
ingest_mssql_server_db_prd_info.columns

['prd_id',
 'prd_key',
 'prd_nm',
 'prd_cost',
 'prd_line',
 'prd_start_dt',
 'prd_end_dt']

In [26]:
ingest_mssql_server_db_prd_info.dtypes

[('prd_id', 'int'),
 ('prd_key', 'string'),
 ('prd_nm', 'string'),
 ('prd_cost', 'int'),
 ('prd_line', 'string'),
 ('prd_start_dt', 'date'),
 ('prd_end_dt', 'date')]

In [27]:
ingest_mssql_server_db_prd_info.show()

+------+----------------+--------------------+--------+--------+------------+----------+
|prd_id|         prd_key|              prd_nm|prd_cost|prd_line|prd_start_dt|prd_end_dt|
+------+----------------+--------------------+--------+--------+------------+----------+
|   210|CO-RF-FR-R92B-58|HL Road Frame - B...|    NULL|      R |  2003-07-01|1900-01-01|
|   211|CO-RF-FR-R92R-58|HL Road Frame - R...|    NULL|      R |  2003-07-01|1900-01-01|
|   212| AC-HE-HL-U509-R|Sport-100 Helmet-...|      12|      S |  2011-07-01|2007-12-28|
|   213| AC-HE-HL-U509-R|Sport-100 Helmet-...|      14|      S |  2012-07-01|2008-12-27|
|   214| AC-HE-HL-U509-R|Sport-100 Helmet-...|      13|      S |  2013-07-01|1900-01-01|
|   215|   AC-HE-HL-U509|Sport-100 Helmet-...|      12|      S |  2011-07-01|2007-12-28|
|   216|   AC-HE-HL-U509|Sport-100 Helmet-...|      14|      S |  2012-07-01|2008-12-27|
|   217|   AC-HE-HL-U509|Sport-100 Helmet-...|      13|      S |  2013-07-01|1900-01-01|
|   218| CL-SO-SO-B90

In [28]:
ingest_mysql_db_sales_info = spark.read.format('jdbc') \
.option('url' , 'jdbc:mysql://mysql_db:3306/DWH_Test') \
.option('dbtable' , 'sales_details') \
.option("driver", "com.mysql.cj.jdbc.Driver") \
.option('user' , 'root') \
.option('password' , 'P@ssw0rd')\
.load()

In [29]:
ingest_mysql_db_sales_info.show(2)

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|    SO43697| BK-R93R-62|      21768|    20101229|   20110105|  20110110|     3578|           1|   3578\r|
|    SO43698| BK-M82S-44|      28389|    20101229|   20110105|  20110110|     3400|           1|   3400\r|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
only showing top 2 rows



In [30]:
ingest_mysql_db_sales_info.dtypes

[('sls_ord_num', 'string'),
 ('sls_prd_key', 'string'),
 ('sls_cust_id', 'string'),
 ('sls_order_dt', 'int'),
 ('sls_ship_dt', 'int'),
 ('sls_due_dt', 'int'),
 ('sls_sales', 'string'),
 ('sls_quantity', 'int'),
 ('sls_price', 'string')]

In [31]:
ingest_mysql_db_sales_info_edit_sales = ingest_mysql_db_sales_info.withColumn('sls_sales' , col('sls_sales').cast(IntegerType()))\
.withColumn('sls_price' , col('sls_price').cast('int'))

In [32]:
ingest_mysql_db_sales_info_edit_sales.dtypes

[('sls_ord_num', 'string'),
 ('sls_prd_key', 'string'),
 ('sls_cust_id', 'string'),
 ('sls_order_dt', 'int'),
 ('sls_ship_dt', 'int'),
 ('sls_due_dt', 'int'),
 ('sls_sales', 'int'),
 ('sls_quantity', 'int'),
 ('sls_price', 'int')]

In [33]:
ingest_mysql_db_sales_info_edit_sales.show(3)

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|    SO43697| BK-R93R-62|      21768|    20101229|   20110105|  20110110|     3578|           1|     3578|
|    SO43698| BK-M82S-44|      28389|    20101229|   20110105|  20110110|     3400|           1|     3400|
|    SO43699| BK-M82S-44|      25863|    20101229|   20110105|  20110110|     3400|           1|     3400|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
only showing top 3 rows



In [34]:
ingest_mssql_server_db_cust_gen = spark.read.format('jdbc') \
.option('url','jdbc:sqlserver://mssql_server:1433;databaseName=Dwhtest;encrypt=true;trustServerCertificate=true') \
.option('dbtable' , 'dbo.cust_gen') \
.option('user','sa')\
.option('password' , 'hazemomar95!')\
.option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
.load()

In [35]:
ingest_mssql_server_db_cust_gen.show(2)

+-------------+----------+------+
|          CID|     BDATE|   GEN|
+-------------+----------+------+
|NASAW00011000|1971-10-06|Male\r|
|NASAW00011001|1976-05-10|Male\r|
+-------------+----------+------+
only showing top 2 rows



In [36]:
ingest_mssql_server_db_cust_gen.show()

+-------------+----------+--------+
|          CID|     BDATE|     GEN|
+-------------+----------+--------+
|NASAW00011000|1971-10-06|  Male\r|
|NASAW00011001|1976-05-10|  Male\r|
|NASAW00011002|1971-02-09|  Male\r|
|NASAW00011003|1973-08-14|Female\r|
|NASAW00011004|1979-08-05|Female\r|
|NASAW00011005|1976-08-01|  Male\r|
|NASAW00011006|1976-12-02|Female\r|
|NASAW00011007|1969-11-06|  Male\r|
|NASAW00011008|1975-07-04|Female\r|
|NASAW00011009|1969-09-29|  Male\r|
|NASAW00011010|1969-08-05|Female\r|
|NASAW00011011|1969-05-03|  Male\r|
|NASAW00011012|1979-01-14|Female\r|
|NASAW00011013|1979-08-03|  Male\r|
|NASAW00011014|1973-11-06|Female\r|
|NASAW00011015|1984-08-26|Female\r|
|NASAW00011016|1984-10-25|  Male\r|
|NASAW00011017|1949-12-24|Female\r|
|NASAW00011018|1955-10-06|  Male\r|
|NASAW00011019|1983-09-04|  Male\r|
+-------------+----------+--------+
only showing top 20 rows



In [37]:
ingest_mssql_server_db_cust_gen_test = ingest_mssql_server_db_cust_gen.withColumn('GEN' , split(col('GEN') , '\r')[0])

In [38]:
ingest_mssql_server_db_cust_gen_test.show(2)

+-------------+----------+----+
|          CID|     BDATE| GEN|
+-------------+----------+----+
|NASAW00011000|1971-10-06|Male|
|NASAW00011001|1976-05-10|Male|
+-------------+----------+----+
only showing top 2 rows



In [39]:
ingest_csv_cust_country = spark.read.format('csv') \
.option('inferSchema' , 'true')\
.option('header' , 'true') \
.option('samplingRatio',0.001) \
.option('path' , '/home/jovyan/work/datasets/source_erp/LOC_A101.csv')\
.load()

In [40]:
ingest_csv_cust_country.printSchema()

root
 |-- CID: string (nullable = true)
 |-- CNTRY: string (nullable = true)



In [41]:
ingest_csv_cust_country.dtypes

[('CID', 'string'), ('CNTRY', 'string')]

In [42]:
ingest_csv_cust_country.show(n=3 , truncate=False)

+-----------+---------+
|CID        |CNTRY    |
+-----------+---------+
|AW-00011000|Australia|
|AW-00011001|Australia|
|AW-00011002|Australia|
+-----------+---------+
only showing top 3 rows



In [43]:
ingest_csv_ped_maintenace = spark.read.format('csv') \
.option('inferSchema' , 'true')\
.option('header' , 'true') \
.option('samplingRatio',0.001) \
.option('path' , '/home/jovyan/work/datasets/source_erp/PX_CAT_G1V2.csv')\
.load()

In [44]:
ingest_csv_ped_maintenace.show(3)

+-----+-----------+-----------------+-----------+
|   ID|        CAT|           SUBCAT|MAINTENANCE|
+-----+-----------+-----------------+-----------+
|AC_BR|Accessories|       Bike Racks|        Yes|
|AC_BS|Accessories|      Bike Stands|         No|
|AC_BC|Accessories|Bottles and Cages|         No|
+-----+-----------+-----------------+-----------+
only showing top 3 rows



## ✅ INGEST THE DATA FROM RDBMS (MYSQL & MSSQL) USING MEDLLIEAN ARCHITECTURE INTO POSTGRESDB

**Take the data from bronz layer(`pyspark`)coming from different sources and puted into `potgresdb`** 

### Puted the `cust_info` table into postgresql 

In [48]:
ingest_mysql_db_edit_date.write.mode('overwrite') \
.format('jdbc')\
.option('url','jdbc:postgresql://pg_db:5432/dwhsliver')\
.option('dbtable' , 'public.cust_info')\
.option('user' ,'postgres')\
.option('password','P@ssw0rd')\
.option("driver", "org.postgresql.Driver")\
.save()

In [50]:
ingest_mysql_db_edit_date.select('*')\
.agg(fn.count('*')).show()

+--------+
|count(1)|
+--------+
|   18491|
+--------+



In [51]:
ingest_mssql_server_db_prd_info.write.mode('overwrite') \
.format('jdbc')\
.option('url','jdbc:postgresql://pg_db:5432/dwhsliver')\
.option('dbtable' , 'public.prd_info')\
.option('user' ,'postgres')\
.option('password','P@ssw0rd')\
.option("driver", "org.postgresql.Driver")\
.save()

In [53]:
ingest_mysql_db_sales_info_edit_sales.write.mode('overwrite') \
.format('jdbc')\
.option('url','jdbc:postgresql://pg_db:5432/dwhsliver')\
.option('dbtable' , 'public.sales_info')\
.option('user' ,'postgres')\
.option('password','P@ssw0rd')\
.option("driver", "org.postgresql.Driver")\
.save()

In [54]:
ingest_mssql_server_db_cust_gen.write.mode('overwrite') \
.format('jdbc')\
.option('url','jdbc:postgresql://pg_db:5432/dwhsliver')\
.option('dbtable' , 'public.cust_gen')\
.option('user' ,'postgres')\
.option('password','P@ssw0rd')\
.option("driver", "org.postgresql.Driver")\
.save()

In [55]:
ingest_csv_cust_country.write.mode('overwrite') \
.format('jdbc')\
.option('url','jdbc:postgresql://pg_db:5432/dwhsliver')\
.option('dbtable' , 'public.cust_country')\
.option('user' ,'postgres')\
.option('password','P@ssw0rd')\
.option("driver", "org.postgresql.Driver")\
.save()

In [56]:
ingest_csv_ped_maintenace.write.mode('overwrite') \
.format('jdbc')\
.option('url','jdbc:postgresql://pg_db:5432/dwhsliver')\
.option('dbtable' , 'public.cust_maintenance')\
.option('user' ,'postgres')\
.option('password','P@ssw0rd')\
.option("driver", "org.postgresql.Driver")\
.save()